# TheAI GatewayPattern

**Module 12 · Lesson 12.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why a Gateway: One Front Door for Every Provider
- Routing and Load Balancing
- Virtual Keys and Auth
- Rate Limiting: Requests AND Tokens
- Fallbacks and Retries as Config
- Cost, Budgets and Observability

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## Five services, five SDKs, one unanswerable question

> 💡 **Info**
>
> Your company has five services that call LLMs, and each one wired up its own provider SDK (one on **OpenAI**, one on **Anthropic**, one on **Google**), its own retry logic, its own API key in an env var, and its own idea of what a call costs. Then three things happen in one week. Finance asks **“which team spent the roughly $40,000 on tokens last month?”** and nobody can answer. A leaked key from one repo has to be **rotated in five places**. And you want to swap the summarizer to a cheaper model - which means a **code change and a deploy in every service**. All three have the same root cause: there is no single place that owns your AI traffic. An **AI gateway** is that place - one OpenAI-compatible front door that centralizes routing, keys, rate limits, fallbacks, cost, and logging as configuration. This lesson builds one from the inside out, then shows you the tools that sell it as a product, and you can run every piece with no API key.

### 🎯 What you will be able to do after this lesson

- **Build** the core gateway layers - a unified facade, a router, virtual-key auth, a token-aware rate limiter, and a cost meter - as runnable code.

- **Compare** routing strategies, request- vs token-based rate limits, and managed vs self-hosted gateways (LiteLLM / Portkey / Cloudflare / Kong / OpenRouter).

- **Operate** a gateway’s config: an order/fallback chain, virtual keys with budgets, and per-team cost attribution.

- **Evaluate** the build-vs-buy decision for an AI gateway against a team’s real needs.

> 📦 **Info**
>
> ✅ Before you startThis turns **12.2**’s hand-coded reliability patterns into gateway *configuration*, and it productizes **11.11**’s credential-proxy idea as virtual keys. **12.1** drew the gateway as a box in the reference architecture; this lesson builds what is inside it. Caching - the next layer a gateway adds - is **Lesson 12.4**, and monitoring the gateway’s logs and cost is **Lesson 12.8**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🛫️ **Analogy**
>
> An AI gateway is an **airport control tower**. Without a tower, every pilot negotiates the runway directly with every other pilot - chaos, and no one has the full picture. The tower is a *single point of coordination*: it routes each plane to a runway, checks credentials, sequences traffic, and logs every movement, so the pilots just talk to one authority instead of to each other. A gateway does exactly that for your AI traffic: your services talk to one front door, and it handles routing, keys, limits, and logging. **Where it breaks down:** a control tower can be a single point of failure - so a real gateway is itself run with redundancy (the reliability patterns from 12.2 apply to the gateway too).

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“I’ll just call each provider SDK directly.”** Then every service re-implements auth, retries, rate limits, cost tracking, and logging - and a model swap is a code change everywhere. A gateway centralizes all of it as config.
> - **“A gateway is just a proxy.”** A proxy forwards bytes. A gateway is the *control plane* for your AI traffic: routing, virtual keys, token-aware limits, fallbacks, cost, and observability, in one place.

> 💡 **Info**
>
> ⚠️ Anti-patternRate-limiting by **request count only**, and handing every app the **real provider key**. A request-only limiter waves through a single 200,000-token call that blows your whole token budget; and the real key, copied into five services, cannot be budgeted or revoked per app and leaks the moment one repo is compromised. Limit by requests AND tokens, and issue scoped virtual keys.

---

## 🎯 Concept 1: Why a Gateway: One Front Door for Every Provider

### Why a Gateway: One Front Door for Every Provider

Wrap every provider behind a single OpenAI-compatible interface, so your app calls one endpoint and a model swap is a config change, not a code change in every service.

Start with the move that makes everything else possible: a **unified facade**. A gateway wraps every provider - OpenAI, Anthropic, Google, and dozens more - behind a **single OpenAI-compatible interface** (meaning it accepts requests and returns responses in the same shape as OpenAI’s chat-completions API - the de-facto format nearly every gateway and provider now speaks - so any OpenAI SDK can point at it unchanged), so your application code makes *one* kind of call and never imports a provider SDK directly. That one indirection is what lets the gateway own the cross-cutting concerns: because every call flows through it, it can add auth, retries, limits, cost, and logging in *one* place instead of in every service. And it collapses the wiring: without a gateway, **N** apps each wire to **M** providers (an N×M tangle of keys and SDKs); with a gateway, it is N apps to one front door to M providers. A **model swap becomes a one-line config change**, not a deploy across every service. The block builds a facade over three providers, keyless.

> 🔐 **Analogy**
>
> It is a **universal power adapter**. Instead of carrying a different plug for every country and every device, you carry one adapter with a single face you plug into, and it handles the wildly different sockets behind it. Your laptop does not know or care which country’s wiring is on the other side. The gateway is that adapter for model providers: your app speaks one ‘plug’ (the OpenAI-compatible call), and the gateway deals with each provider’s different API behind it.

Without a gateway, 6 apps each call 4 providers directly. How many app-to-provider integrations must you maintain?

**📝 `01_gateway_facade.py`** - *runnable*

In [ ]:
# ONE FRONT DOOR - a gateway wraps every provider behind a SINGLE OpenAI-compatible interface,
# so your app calls one function and a model swap becomes a config change, not a code change.
# Scripted adapters stand in for the real SDKs, so this runs with no key.

# Each provider adapter normalizes its own API into the SAME response shape.
def openai_adapter(model, prompt):    return {"text": f"[openai:{model}] {prompt[:18]}...", "provider": "openai"}
def anthropic_adapter(model, prompt): return {"text": f"[anthropic:{model}] {prompt[:18]}...", "provider": "anthropic"}
def gemini_adapter(model, prompt):    return {"text": f"[gemini:{model}] {prompt[:18]}...", "provider": "gemini"}

# The gateway's model -> provider map is CONFIG (not code).
MODEL_ROUTES = {
    "gpt-5":             openai_adapter,
    "claude-sonnet-4-6": anthropic_adapter,
    "gemini-2.5-flash":  gemini_adapter,
}

def gateway_chat(model, prompt):          # the ONE call every app makes
    adapter = MODEL_ROUTES[model]
    return adapter(model, prompt)

print("Three providers, one OpenAI-compatible call:")
for model in ["gpt-5", "claude-sonnet-4-6", "gemini-2.5-flash"]:
    r = gateway_chat(model, "Summarize the refund policy")
    print(f"  gateway_chat({model:18}) -> provider={r['provider']:9} {r['text']}")

# A model swap is a one-line config change - the app code never changes.
DEFAULT_MODEL = "gpt-5"
print(f"\nApp calls gateway_chat(DEFAULT_MODEL): default is {DEFAULT_MODEL}")
DEFAULT_MODEL = "gemini-2.5-flash"        # swap to a cheaper model in CONFIG
print(f"Swap DEFAULT_MODEL -> {DEFAULT_MODEL}: same app code, now routed to {gateway_chat(DEFAULT_MODEL, 'hi')['provider']}")
# Output:
# Three providers, one OpenAI-compatible call:
#   gateway_chat(gpt-5             ) -> provider=openai    [openai:gpt-5] Summarize the refu...
#   gateway_chat(claude-sonnet-4-6 ) -> provider=anthropic [anthropic:claude-sonnet-4-6] Summarize the refu...
#   gateway_chat(gemini-2.5-flash  ) -> provider=gemini    [gemini:gemini-2.5-flash] Summarize the refu...
#
# App calls gateway_chat(DEFAULT_MODEL): default is gpt-5
# Swap DEFAULT_MODEL -> gemini-2.5-flash: same app code, now routed to gemini

- `MODEL_ROUTES` maps a model name to a provider adapter - the routing table is **config**, not code.
- Each adapter normalizes its provider’s API into the **same response shape**, so the app sees one interface.
- `gateway_chat(model, prompt)` is the **one call** every app makes, whichever provider ultimately serves it.
- Swapping `DEFAULT_MODEL` re-routes to a different provider with **no change to the app code** - the whole point.

#### 💡 What just happened

⚡ What just happened? A gateway wraps every provider behind one OpenAI-compatible facade, so the app makes one call and a model swap is a config change. Tradeoff: the indirection adds a hop and a component you must run reliably, but it is what lets you own auth, routing, limits, cost, and logging in one place instead of N. One honest simplification: these scripted adapters normalize only the RESPONSE shape to stay keyless - a production facade must also translate the REQUEST across providers (system-prompt placement, parameter names, and tool-calling and structured-output formats differ between OpenAI, Anthropic, and Google). Every later step is a concern this front door now centralizes.

- Without a gateway, N apps wire to M providers - an NxM tangle
- With a gateway, N apps -> one front door -> M providers; a model swap flips in one place

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Routing and Load Balancing

### Routing and Load Balancing

A gateway spreads a request across a model group by a strategy - priority order, weight, or lowest latency - and pulls a failing deployment out of rotation with a cooldown.

Once every call flows through the front door, the gateway can **load-balance** it. A **model group** is several deployments of the same logical model - the same model on two different keys, or across OpenAI and Azure - and the router picks one by a **strategy**: **priority order** (a ranked list; try the top one first), **weighted** (send a bigger share to the higher-weighted deployment), or **least-latency** (pick the fastest right now). LiteLLM offers exactly these - `simple-shuffle`, `least-busy`, `usage-based-routing`, `latency-based-routing`. And when a deployment returns a **429** (rate-limited), the gateway places it on a **cooldown** so traffic shifts off it until it recovers. The block routes one request three ways and then trips a cooldown.

> 🏨 **Analogy**
>
> It is a **hotel concierge routing guests to lifts**. When you ask for your floor, the concierge does not send everyone to the same lift - they send you to the nearest free one, or spread the crowd across several, and if a lift breaks down they stop directing people to it until it is fixed. The router is that concierge for your requests: it spreads them across equivalent deployments by a strategy and quietly pulls a broken one out of rotation.

Deployments: A (priority 1, slow), B (highest weight), C (lowest latency). Which does a LEAST-LATENCY strategy pick?

**📝 `02_router.py`** - *runnable*

In [ ]:
# ROUTING & LOAD BALANCING - a gateway routes a request across a MODEL GROUP (several deployments
# of the same logical model) by a strategy: priority order, weighted, or least-latency. A 429 puts a
# deployment on cooldown so traffic shifts off it. Deterministic, no key.

# A model group: deployments with an order (lower = higher priority), a weight, and a latency (ms).
deployments = [
    {"id": "openai-key-1",  "order": 1, "weight": 1, "latency": 800, "cooling": False},
    {"id": "azure-key-2",   "order": 2, "weight": 5, "latency": 500, "cooling": False},
    {"id": "openai-key-3",  "order": 3, "weight": 1, "latency": 200, "cooling": False},
]
def available(deps): return [d for d in deps if not d["cooling"]]

def route(deps, strategy):
    live = available(deps)
    if strategy == "priority":   return min(live, key=lambda d: d["order"])
    if strategy == "weighted":   return max(live, key=lambda d: d["weight"])   # highest weight = biggest share
    if strategy == "latency":    return min(live, key=lambda d: d["latency"])

print("Same request, three routing strategies pick three different deployments:")
for s in ["priority", "weighted", "latency"]:
    print(f"  {s:9} -> {route(deployments, s)['id']}")

# A 429 on the top-priority deployment puts it on COOLDOWN; priority then shifts to the next.
print("\nopenai-key-1 returns 429 -> placed on cooldown:")
deployments[0]["cooling"] = True
print(f"  priority now -> {route(deployments, 'priority')['id']}  (traffic shifted off the cooling deployment)")
# Output:
# Same request, three routing strategies pick three different deployments:
#   priority  -> openai-key-1
#   weighted  -> azure-key-2
#   latency   -> openai-key-3
#
# openai-key-1 returns 429 -> placed on cooldown:
#   priority now -> azure-key-2  (traffic shifted off the cooling deployment)

- The same request routes to **three different deployments** under the three strategies - priority picks A, weighted picks the heavy-weighted B, least-latency picks the fast C.
- The strategy is a **config choice**, not app code - you change how traffic spreads without touching any service.
- When `openai-key-1` returns a 429, it is marked `cooling` and dropped from the live set.
- Priority then shifts to the next available deployment automatically - the cooldown is how the gateway routes around a rate-limited backend.

#### 💡 What just happened

⚡ What just happened? A gateway routes each request across a model group by a strategy (priority, weighted, least-latency) and cools down a deployment that returns a 429. Tradeoff: more deployments and keys to manage, but you get horizontal headroom (more combined RPM/TPM) and automatic routing around a throttled backend. Which strategy to use is a config line, not a rewrite.

- A request routes a model group by strategy: priority / weighted / least-latency
- A 429 puts a deployment on cooldown and traffic shifts to the next

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Virtual Keys and Auth

### Virtual Keys and Auth

The gateway holds the real provider keys and issues each app a scoped virtual key with its own model allow-list and budget, so the real keys never leave the gateway.

Now the security move. The naive setup puts the **real provider key** in every service’s env vars - which means the key is copied everywhere, cannot be budgeted or rate-limited per app, and leaks the instant any one repo is compromised. A gateway fixes this with **virtual keys**: the gateway holds the real provider keys in *one* place and issues each team or app a **scoped virtual key** with its own **model allow-list**, **budget**, and rate limits. A request is authorized against the virtual key - wrong model, over budget, or unknown key gets rejected before any real call - and you can revoke or re-budget one app without touching the others. In LiteLLM you set a `master_key` and mint keys via `POST /key/generate`, each with its own `tpm_limit` and `rpm_limit`. The block authorizes three requests against two virtual keys.

> 🏩️ **Analogy**
>
> It is **visitor badges at a secure building**. Visitors do not get a copy of the master key that opens every door - they get a temporary badge scoped to the floors they are allowed on, with an expiry, and security can deactivate one badge without re-keying the building. The gateway is the front desk: it keeps the master keys in the back and hands out scoped badges (virtual keys) to each app, so a lost badge is a five-second revocation, not a building-wide lock change.

An app presents a virtual key scoped to gemini-2.5-flash and asks the gateway to call gpt-5. What happens?

**📝 `03_virtual_keys.py`** - *runnable*

In [ ]:
# VIRTUAL KEYS - the gateway holds the REAL provider keys and hands each app a scoped VIRTUAL key
# with its own model allow-list and budget. A request is authorized against the virtual key; the real
# keys never leave the gateway. Scripted, deterministic, no key.

# The gateway's secret store (real keys) - NEVER returned to any app.
_REAL_KEYS = {"openai": "sk-real-openai-...", "anthropic": "sk-real-anthropic-..."}

# Virtual keys issued to teams: what each is allowed to do.
virtual_keys = {
    "vk-team-support": {"allowed": {"gpt-5", "gemini-2.5-flash"}, "budget_left": 5.00},
    "vk-team-intern":  {"allowed": {"gemini-2.5-flash"},          "budget_left": 0.00},
}

def authorize(vkey, model, est_cost):
    k = virtual_keys.get(vkey)
    if k is None:                    return False, "unknown virtual key"
    if model not in k["allowed"]:    return False, f"model {model} not in this key's allow-list"
    if est_cost > k["budget_left"]:  return False, f"over budget (${k['budget_left']:.2f} left)"
    return True, "authorized"

print("Authorize requests against virtual keys (real provider keys stay in the gateway):")
tests = [("vk-team-support", "gpt-5", 0.20),          # allowed + in budget
         ("vk-team-support", "claude-sonnet-4-6", 0.20),  # model not allowed
         ("vk-team-intern",  "gemini-2.5-flash", 0.20)]   # over budget ($0 left)
for vkey, model, cost in tests:
    ok, why = authorize(vkey, model, cost)
    print(f"  {vkey:16} {model:18} -> {'PASS' if ok else 'REJECT'} ({why})")
print(f"\nThe real keys ({list(_REAL_KEYS)}) were never exposed to any app - only virtual keys are.")
# Output:
# Authorize requests against virtual keys (real provider keys stay in the gateway):
#   vk-team-support  gpt-5              -> PASS (authorized)
#   vk-team-support  claude-sonnet-4-6  -> REJECT (model claude-sonnet-4-6 not in this key's allow-list)
#   vk-team-intern   gemini-2.5-flash   -> REJECT (over budget ($0.00 left))
#
# The real keys (['openai', 'anthropic']) were never exposed to any app - only virtual keys are.

- `_REAL_KEYS` lives only in the gateway and is **never returned to any app** - apps only ever hold virtual keys.
- The support key calling an allowed model within budget is **authorized**; the same key calling a non-allow-listed model is **rejected**.
- The intern key is **rejected for being over budget** (zero left) - the budget is enforced at auth time, before any provider call.
- Revoking or re-budgeting one virtual key touches no other app - the isolation the shared-real-key setup can never give you.

#### 💡 What just happened

⚡ What just happened? The gateway holds the real keys and issues scoped virtual keys per app, enforcing a model allow-list and budget at auth time. Tradeoff: you run a key-management layer (with a database), but you get per-app budgets, rate limits, and one-click revocation, and the real keys never leave the gateway. This is 11.11’s credential proxy, productized.

- Apps hold scoped virtual keys; the gateway holds the real provider keys
- A scoped key passes an allowed model; an out-of-scope or over-budget key is rejected

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Rate Limiting: Requests AND Tokens

### Rate Limiting: Requests AND Tokens

LLM limits are token-based, not just request-based. A request-only limiter waves through a single long-context call that blows the whole token budget, so a gateway enforces RPM AND TPM.

Here is the rate-limiting trap that catches teams coming from ordinary web APIs. A normal API limits **requests per minute** (RPM) - a raw call count. But LLM providers **charge by the token**, so they also limit **tokens per minute** (TPM), and the TPM limit is the one that actually protects capacity and cost. Why? Because one call is not like another: a single **200,000-token** context call costs about as much as *fifty* ordinary four-thousand-token calls, and it can consume most of a TPM budget on its own. A request-only limiter sees that whale as *one request* and waves it straight through. So a gateway must enforce **both** axes - RPM to stop a burst of tiny calls, TPM to stop a whale. The block trips RPM with a burst, then shows a request-only limiter missing a whale that the token-aware limiter catches.

> 🚛 **Analogy**
>
> It is a **highway weigh station, not just a car counter**. A simple traffic gate counts vehicles - but a single 40-ton truck stresses the bridge far more than fifty cars, and a car-counter waves it through because it is ‘one vehicle.’ A weigh station checks the *weight*, not just the count, and stops the overloaded truck. Token-aware rate limiting is the weigh station: it measures how heavy each call is (its tokens), not just how many calls there are.

Your gateway limits only requests per minute. A single 200,000-token call arrives. Does the limiter stop it?

**📝 `04_rate_limit.py`** - *runnable*

In [ ]:
# RATE LIMITING - LLM limits are on TWO axes: requests-per-minute (RPM) AND tokens-per-minute (TPM).
# Token limits are the load-bearing ones: one long-context call can blow the TPM budget while sailing
# past a request-only limiter. A gateway must enforce BOTH. Deterministic, no key.

RPM_LIMIT, TPM_LIMIT = 4, 100_000
reqs, toks = 0, 0

def allow(n_tokens, token_aware=True):
    global reqs, toks
    reqs += 1; toks += n_tokens
    if reqs > RPM_LIMIT:                        return False, f"RPM exceeded ({reqs} > {RPM_LIMIT})"
    if token_aware and toks > TPM_LIMIT:        return False, f"TPM exceeded ({toks:,} > {TPM_LIMIT:,})"
    return True, "allowed"

print("A burst of 5 small calls (500 tokens each) trips the RPM limit:")
for i in range(5):
    ok, why = allow(500)
    print(f"  call {i+1}: 500 tokens -> {'allowed' if ok else 'BLOCKED: ' + why}")

# Reset, then send ONE 200,000-token 'whale' call.
reqs, toks = 0, 0
print("\nOne 200,000-token call - the request-only limiter MISSES it, the token-aware one catches it:")
ok_req, _ = allow(200_000, token_aware=False)      # RPM-only: 1 request, looks fine
reqs, toks = 0, 0
ok_tok, why = allow(200_000, token_aware=True)      # RPM + TPM: the tokens blow the budget
print(f"  request-only limiter: {'allowed' if ok_req else 'BLOCKED'}  <- BUG: 1 call = 1 request, so it passes")
print(f"  token-aware limiter:  {'allowed' if ok_tok else 'BLOCKED'}  <- {why} (one whale = 2x the entire TPM budget)")
# Output:
# A burst of 5 small calls (500 tokens each) trips the RPM limit:
#   call 1: 500 tokens -> allowed
#   call 2: 500 tokens -> allowed
#   call 3: 500 tokens -> allowed
#   call 4: 500 tokens -> allowed
#   call 5: 500 tokens -> BLOCKED: RPM exceeded (5 > 4)
#
# One 200,000-token call - the request-only limiter MISSES it, the token-aware one catches it:
#   request-only limiter: allowed  <- BUG: 1 call = 1 request, so it passes
#   token-aware limiter:  BLOCKED  <- TPM exceeded (200,000 > 100,000) (one whale = 2x the entire TPM budget)

- A **burst of five small calls** trips the RPM limit at call five - the request axis catches a flood of tiny calls.
- The **200,000-token whale** is one request, so the **request-only limiter passes it** (the bug) - and it would blow the token budget.
- The **token-aware limiter blocks it**: the tokens exceed the TPM budget, even though it is a single request.
- The lesson: enforce RPM *and* TPM, because the two axes catch different abuses - a burst of small calls, and one enormous one.

#### 💡 What just happened

⚡ What just happened? LLM rate limits are two-dimensional: requests per minute (a burst of tiny calls) and tokens per minute (one long-context whale). Tradeoff: tracking tokens per key needs a fast counter (Redis in production), but a request-only limiter cannot protect a token budget. A gateway enforces both, per virtual key. One streaming nuance: for a streamed (SSE) response the output token count is known only when the stream closes, so the gateway admits the request against TPM using the input tokens plus an output estimate, then reconciles the actual tokens and cost at stream close.

- Two meters fill: requests-per-minute and tokens-per-minute
- A burst of small calls trips RPM; one 200k-token whale trips TPM while RPM is fine

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Fallbacks and Retries as Config

### Fallbacks and Retries as Config

The reliability patterns from 12.2 become declarative gateway config - an order chain plus cross-model fallbacks in one versioned file - so every app is protected without per-service retry code.

In 12.2 you hand-coded a resilient client. A gateway lets you write that *once*, as **configuration**. The failover rules live in a **versioned config file**: within a model group, an **order chain** (try the order-1 deployment, then order-2); and when the whole group is exhausted, a **cross-model fallback** to a different model entirely. Add `num_retries` and a `timeout`, and the gateway applies the reliability spine from 12.2 to *every* app behind it - no retry code sprinkled through each service, and changing the failover behavior is a config edit, not a redeploy of five services. The block drives failover purely from a config object: healthy, a 429 on the primary, then the whole group down.

> 🔌 **Analogy**
>
> It is a **universal adapter with a spare socket wired in**. You do not teach every appliance in the house what to do when the main outlet dies - you wire the fallback once, at the adapter: if the primary socket has no power, it automatically switches to the backup line. The gateway config is that wiring: the fallback chain is defined once at the front door, and every app plugged into it inherits the failover for free.

The whole gpt-5 model group is down. The config lists a cross-model fallback to claude-sonnet-4-6. What serves the request?

**📝 `05_fallbacks_config.py`** - *runnable*

In [ ]:
# FALLBACKS AS CONFIG - 12.2's reliability patterns become DECLARATIVE gateway config: an order chain
# within a model group, plus cross-model fallbacks, in a versioned config file. One config protects
# every app - no per-app retry code. Scripted, deterministic, no key.

# The gateway config (what you version in your repo, not code sprinkled through every service).
CONFIG = {
    "model_groups": {
        "chat": [
            {"deployment": "gpt-5@openai",  "order": 1},
            {"deployment": "gpt-5@azure",   "order": 2},   # same model, second key
        ]
    },
    "fallbacks": {"chat": ["claude-sonnet-4-6"]},           # cross-model fallback when the group is exhausted
}

def call(deployment, down):  return "ok" if deployment not in down else "429"

def route_with_fallback(config, group, down):
    for dep in sorted(config["model_groups"][group], key=lambda d: d["order"]):   # try the order chain
        if call(dep["deployment"], down) == "ok":
            return dep["deployment"], "primary group"
    for fb in config["fallbacks"].get(group, []):                                  # then cross-model fallbacks
        if call(fb, down) == "ok":
            return fb, "cross-model fallback"
    return None, "all failed"

print("Config-driven failover (no per-app code):")
served, via = route_with_fallback(CONFIG, "chat", down=set())
print(f"  everything healthy      -> served by {served} ({via})")
served, via = route_with_fallback(CONFIG, "chat", down={"gpt-5@openai"})
print(f"  gpt-5@openai 429        -> served by {served} ({via})")
served, via = route_with_fallback(CONFIG, "chat", down={"gpt-5@openai", "gpt-5@azure"})
print(f"  whole gpt-5 group down  -> served by {served} ({via})")
print("\nChanging the failover behavior is a CONFIG edit, not a redeploy of every service.")
# Output:
# Config-driven failover (no per-app code):
#   everything healthy      -> served by gpt-5@openai (primary group)
#   gpt-5@openai 429        -> served by gpt-5@azure (primary group)
#   whole gpt-5 group down  -> served by claude-sonnet-4-6 (cross-model fallback)
#
# Changing the failover behavior is a CONFIG edit, not a redeploy of every service.

- `CONFIG` is the versioned file: a model group with an **order chain** and a **cross-model fallback** - declarative, not code.
- Healthy: served by the order-1 deployment (`gpt-5@openai`).
- A 429 on the primary: the gateway moves to **order-2 in the same group** (`gpt-5@azure`) - no app code ran.
- The whole gpt-5 group down: it falls through to the **cross-model fallback** (`claude-sonnet-4-6`) - the reliability spine of 12.2, applied by config to every app.

#### 💡 What just happened

⚡ What just happened? Fallbacks and retries become declarative gateway config (an order chain plus cross-model fallbacks) in one versioned file, applied to every app. Tradeoff: the failover is centralized (so the gateway must be reliable itself), but you stop copy-pasting retry logic into every service and can change the behavior with a config edit. This is 12.2 as configuration.

- A config file (order chain + cross-model fallbacks) drives failover live
- Edit the config and the routing changes with no app code touched

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Cost, Budgets and Observability

### Cost, Budgets and Observability

The gateway meters every call as tokens times price, attributes it per team, enforces a budget, and logs a structured line - one place that answers who spent what across every provider.

Because every call flows through the gateway, it is the one place that can **meter cost**. It computes each call as **tokens times the model’s price**, attributes it **per virtual key, team, and user** across all providers, and enforces a **budget** - a call that would push a team over its monthly cap is rejected before it runs. And it emits a **structured log line** for every request (model, tokens, cost, latency, which fallback fired), so cost and observability live in one pane of glass instead of five provider dashboards. This is the answer to the cold-open’s unanswerable question: with a gateway, “which team spent what” is a query, not a forensic reconstruction. The block meters four calls, enforces the budget, and logs each one.

> 📈 **Analogy**
>
> It is an **itemized utility meter with a monthly cap**. Instead of a single mystery bill at the end of the month, the meter records every unit as it is used, tags it to the room that used it, and can cut off supply when a budget is hit. The gateway is that meter for tokens: it records each call’s cost as it happens, tags it to the team, and stops a team that has blown its cap - so the bill is never a surprise and never unattributable.

A team has spent roughly 90 percent of its monthly budget. A call that would push the running total over the cap arrives. What does the gateway do?

**📝 `06_cost_budget.py`** - *runnable*

In [ ]:
# COST, BUDGETS & OBSERVABILITY - the gateway meters every call as tokens x price, attributes it per
# team, enforces a budget, and logs a structured line. One place for spend across all providers.
# Prices are illustrative ($/1k tokens). Deterministic, no key.

PRICE = {"gpt-5": 0.005, "claude-sonnet-4-6": 0.003, "gemini-2.5-flash": 0.0004}   # $ per 1k tokens (illustrative)
team_spend = {"support": 0.0}
TEAM_BUDGET = 1.00                                                                  # monthly cap (illustrative)

def meter(team, model, tokens):
    cost = round(tokens / 1000 * PRICE[model], 4)
    if team_spend[team] + cost > TEAM_BUDGET:
        return None, f"BLOCKED: team '{team}' would exceed its ${TEAM_BUDGET:.2f} budget"
    team_spend[team] += cost
    log = {"team": team, "model": model, "tokens": tokens, "cost": cost, "spend_after": round(team_spend[team], 4)}
    return log, "ok"

print("Meter each call, attribute per team, enforce the budget, log a structured line:")
calls = [("support", "gpt-5", 60_000), ("support", "gpt-5", 60_000),
         ("support", "gpt-5", 60_000), ("support", "gpt-5", 60_000)]   # 4 x $0.30 = $1.20 > $1.00
for team, model, tokens in calls:
    log, status = meter(team, model, tokens)
    if log:
        print(f"  {log}")
    else:
        print(f"  {status}  (spend so far ${team_spend[team]:.2f})")
print(f"\nOne dashboard answers 'who spent what': support = ${team_spend['support']:.2f}, and the 4th call was denied.")
# Output:
# Meter each call, attribute per team, enforce the budget, log a structured line:
#   {'team': 'support', 'model': 'gpt-5', 'tokens': 60000, 'cost': 0.3, 'spend_after': 0.3}
#   {'team': 'support', 'model': 'gpt-5', 'tokens': 60000, 'cost': 0.3, 'spend_after': 0.6}
#   {'team': 'support', 'model': 'gpt-5', 'tokens': 60000, 'cost': 0.3, 'spend_after': 0.9}
#   BLOCKED: team 'support' would exceed its $1.00 budget  (spend so far $0.90)
#
# One dashboard answers 'who spent what': support = $0.90, and the 4th call was denied.

- Each call is metered as **tokens x price** and added to the team’s running **spend**, with a structured log line per call.
- The first three calls are allowed and logged; the running total climbs toward the budget.
- The fourth call would push the team **over its budget**, so it is **blocked before it runs** - no overrun.
- One dashboard now answers “who spent what” across every provider - the cold-open’s question, answered by the gateway.

#### 💡 What just happened

⚡ What just happened? The gateway meters every call (tokens x price), attributes spend per team, enforces budgets, and logs a structured trace across all providers. Tradeoff: you maintain a price table and a spend store, but you get real-time per-team cost attribution, hard budget caps, and one observability pane. The deep eval/observability stack it feeds is Module 14.

- Calls meter tokens x price into per-team spend bars; a structured log streams
- A team hits its budget cap and the next call is blocked

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Landscape: Build vs Buy

### The Landscape: Build vs Buy

Every gateway is self-hosted (you own it) or managed (zero ops). Match the tool to the team: LiteLLM, Portkey, Cloudflare AI Gateway, Kong, or OpenRouter.

You have now built a gateway from the inside, so you understand exactly what the products do. The 2026 landscape splits on **self-hosted vs managed**. **LiteLLM** is the default open-source, **self-hosted** proxy - a Docker container plus a small Postgres, OpenAI-compatible over 100+ providers, with budgets per key; guardrails and semantic caching exist but are not its focus. **Portkey** adds **guardrails and semantic caching** and rich observability, mostly as SaaS. **Cloudflare AI Gateway** is **fully managed, near-zero ops** with edge caching - ideal if you already live on Cloudflare. **Kong AI Gateway** brings LLM routing into an existing **Kong** API mesh with SSO and PII redaction. And **OpenRouter** is a **managed unified API** giving you one key over 300+ models. The decision is **build vs buy**: self-host for control, buy for guardrails, caching, or zero ops. The block routes a team’s needs to a tool; the illustrative block shows a real LiteLLM config.

> 📦 **Analogy**
>
> It is **choosing how to ship packages**. You can run your own delivery fleet (self-host LiteLLM) - total control, but you buy the vans and hire the drivers. Or you hand it to a courier (Portkey, Cloudflare, OpenRouter) - less control, but they own the trucks, the tracking, and the 2am problems. Neither is ‘right’ - a business shipping millions of parcels runs its own fleet; a shop shipping a few buys postage. You pick by how much control and how much ops burden you actually want.

A team needs built-in guardrails and semantic caching and does not want to run infrastructure. Which gateway fits best?

**📝 `07_landscape.py`** - *runnable*

In [ ]:
# THE LANDSCAPE - build vs buy. A gateway is either self-hosted (you own it) or managed (zero ops).
# Route a team's needs to the right tool. Deterministic, no key.

def choose(self_host, needs_guardrails_or_semantic_cache, has_kong_mesh, in_cloudflare, wants_one_key):
    if has_kong_mesh:                         return "Kong AI Gateway (LLM routing inside your existing Kong mesh)"
    if needs_guardrails_or_semantic_cache:    return "Portkey (guardrails + semantic caching + observability, mostly SaaS)"
    if in_cloudflare:                         return "Cloudflare AI Gateway (fully managed, edge caching, zero ops)"
    if self_host:                             return "LiteLLM (self-hosted OpenAI-compatible proxy, budgets per key)"
    if wants_one_key:                         return "OpenRouter (managed unified API, one key over 300+ models)"
    return "LiteLLM (sensible open-source default)"

cases = [
    ("full control, self-host everything",        dict(self_host=True,  needs_guardrails_or_semantic_cache=False, has_kong_mesh=False, in_cloudflare=False, wants_one_key=False)),
    ("need guardrails + semantic caching",        dict(self_host=False, needs_guardrails_or_semantic_cache=True,  has_kong_mesh=False, in_cloudflare=False, wants_one_key=False)),
    ("already run a Kong API mesh",               dict(self_host=False, needs_guardrails_or_semantic_cache=False, has_kong_mesh=True,  in_cloudflare=False, wants_one_key=False)),
    ("live in Cloudflare, want zero ops",         dict(self_host=False, needs_guardrails_or_semantic_cache=False, has_kong_mesh=False, in_cloudflare=True,  wants_one_key=False)),
    ("just want one key over every model",        dict(self_host=False, needs_guardrails_or_semantic_cache=False, has_kong_mesh=False, in_cloudflare=False, wants_one_key=True)),
]
print("Route each team's needs to a gateway:")
for name, props in cases:
    print(f"  {name:42} -> {choose(**props)}")
print("\nBuild vs buy: self-host for control (LiteLLM); buy for guardrails/caching/zero-ops (Portkey/Cloudflare/OpenRouter).")
# Output:
# Route each team's needs to a gateway:
#   full control, self-host everything         -> LiteLLM (self-hosted OpenAI-compatible proxy, budgets per key)
#   need guardrails + semantic caching         -> Portkey (guardrails + semantic caching + observability, mostly SaaS)
#   already run a Kong API mesh                -> Kong AI Gateway (LLM routing inside your existing Kong mesh)
#   live in Cloudflare, want zero ops          -> Cloudflare AI Gateway (fully managed, edge caching, zero ops)
#   just want one key over every model         -> OpenRouter (managed unified API, one key over 300+ models)
#
# Build vs buy: self-host for control (LiteLLM); buy for guardrails/caching/zero-ops (Portkey/Cloudflare/OpenRouter).

- `choose()` routes by the team’s real needs - an existing Kong mesh, guardrails/semantic-cache, the Cloudflare ecosystem, full control, or one-key simplicity.
- Self-host **LiteLLM** when you want control and data residency; buy **Portkey** for guardrails and semantic caching.
- **Cloudflare** for zero ops on the edge; **Kong** when you already run its mesh; **OpenRouter** for one key over everything.
- The illustrative block below shows the real thing: a LiteLLM `config.yaml` that centralizes everything you built by hand.

**📝 `07b_litellm_config.py`** - *illustrative*

In [ ]:
# THE PRODUCTION GATEWAY - a LiteLLM config.yaml (illustrative minimal example).
# You do not hand-roll the gateway in production; you run LiteLLM (or Portkey / Cloudflare) and put
# routing, keys, limits, and fallbacks in ONE versioned config file. App code just calls the proxy.

# config.yaml
CONFIG = """
model_list:                                    # a model GROUP: two deployments of one logical model
  - model_name: chat
    litellm_params: {model: openai/gpt-5, api_key: os.environ/OPENAI_KEY, rpm: 500}
  - model_name: chat
    litellm_params: {model: azure/gpt-5, api_key: os.environ/AZURE_KEY, rpm: 500}

router_settings:
  routing_strategy: latency-based-routing       # or simple-shuffle / least-busy / usage-based-routing
  num_retries: 3
  fallbacks: [{chat: [claude-sonnet-4-6]}]       # cross-model fallback when the group is exhausted

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY      # mint scoped virtual keys via POST /key/generate
"""
# Start the proxy:   litellm --config config.yaml         (a Docker container + a small Postgres)
# Apps call the OpenAI-compatible endpoint with a VIRTUAL key, never a real provider key:
#   from openai import OpenAI
#   client = OpenAI(base_url="http://gateway:4000", api_key="sk-virtual-team-support")
#   client.chat.completions.create(model="chat", messages=[...])
# Output: (illustrative minimal example - needs `pip install litellm[proxy]` + provider keys in env.)
# One config.yaml centralizes routing, retries, fallbacks, virtual keys, and budgets for every app.

- `model_list` declares a **model group** (two deployments of `chat`) - Step 2’s routing, as config.
- `router_settings` sets the **routing_strategy**, `num_retries`, and **fallbacks** - Steps 2 and 5, in three lines.
- `general_settings.master_key` mints **scoped virtual keys** via `/key/generate` - Step 3.
- Apps call the **OpenAI-compatible endpoint** with a virtual key, never a real provider key - everything you built by hand, in one versioned file.

#### 💡 What just happened

⚡ What just happened? Every gateway is self-hosted (LiteLLM, full control) or managed (Portkey / Cloudflare / OpenRouter, zero ops), and you pick by need: guardrails and semantic caching -> Portkey, an existing Kong mesh -> Kong, the Cloudflare edge -> Cloudflare. Tradeoff / the whole lesson: build to own the control plane and your data, buy to skip the ops and get guardrails/caching for free. Caching itself is the next layer, in Lesson 12.4.

- Toggle needs: self-host? guardrails? semantic cache? Kong mesh? zero-ops?
- The decision routes to LiteLLM / Portkey / Cloudflare / Kong / OpenRouter

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> An AI gateway is the **control plane** for your AI traffic. A **unified facade** puts every provider behind one OpenAI-compatible front door, so a model swap is a config change (Step 1). Behind it, the gateway **routes and load-balances** across a model group and cools down a throttled backend (Step 2); issues **scoped virtual keys** so the real keys never leave and every app has its own budget (Step 3); enforces **rate limits on requests AND tokens** so a whale call cannot blow the budget (Step 4); applies **fallbacks and retries as config** so 12.2’s reliability spine protects every app (Step 5); and **meters cost, enforces budgets, and logs** every call so “who spent what” is one query (Step 6). Finally, you **build or buy** - LiteLLM to self-host, Portkey / Cloudflare / OpenRouter to go managed (Step 7). Five services, five SDKs, one unanswerable question becomes one front door that owns it all.

| Gateway | Hosting | Standout feature | Best for |
|---|---|---|---|
| LiteLLM | self-hosted (Docker + Postgres) | OpenAI-compatible over 100+ providers; budgets per key | full control, data residency |
| Portkey | mostly managed (SaaS) | guardrails + semantic caching + observability | governance without the ops |
| Cloudflare AI Gateway | fully managed | edge caching + analytics, near-zero ops | already on Cloudflare |
| Kong AI Gateway | self-hosted mesh | LLM routing in a Kong mesh; SSO, PII redaction | you already run Kong |
| OpenRouter | managed | one key over 300+ models | minimal setup, one bill |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now put a control plane in front of every provider. The next layer a gateway adds is **caching** (prompt, semantic, and edge) - that is **Lesson 12.4**, where Portkey’s and Cloudflare’s semantic caching gets its own treatment. **Autoscaling** the gateway and its backends is **Lesson 12.7**, monitoring its structured logs and per-team cost alerts is **Lesson 12.8**, and the deep observability and eval stack it feeds (LangSmith, Langfuse, OpenTelemetry `gen_ai.*`) is **Module 14**. Guardrails and PII filtering - the other control-plane feature a gateway can host - you met with agents in **Lesson 11.11** and go deeper on in **Module 15**. The primary references are the [LiteLLM proxy](https://github.com/BerriAI/litellm), its [routing docs](https://docs.litellm.ai/docs/routing), [Portkey](https://portkey.ai/), [Cloudflare AI Gateway](https://developers.cloudflare.com/ai-gateway/), and **OpenRouter**.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **TheAI GatewayPattern**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.3.html` - regenerate this notebook whenever the source HTML is updated.*
